In [ ]:
import sys
sys.path.insert(0, '../code')
import paths  # input and output locations, from config.yaml
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '../code')
from constants import CONTINENT_COLORS, REGION_TO_CONTINENT

In [ ]:
# Pixel-level socioeconomic data (covers populated areas where GDP data is available)
df = pd.read_csv(paths.der("travel_socio_eco_class.csv"))

THRESHOLDS = [1, 5, 10, 15, 30, 45, 90, 120, 180]
INCOME_ORDER = ['High', 'Upper-middle', 'Lower-middle', 'Low']
INCOME_COLORS = {
    'High':          '#2166ac',
    'Upper-middle':  '#4dac26',
    'Lower-middle':  '#e08214',
    'Low':           '#d73027'
}
INCOME_LABELS = {
    'High':          'High income',
    'Upper-middle':  'Upper-middle income',
    'Lower-middle':  'Lower-middle income',
    'Low':           'Low income'
}

# Cumulative % within each income group
rows = []
for grp, g in df.groupby('income_group'):
    tot = g['population'].sum()
    for t in THRESHOLDS:
        within = g[g['travel_time'] <= t]['population'].sum()
        rows.append({'income_group': grp, 'threshold_min': t, 'pct': 100 * within / tot})
df_pct = pd.DataFrame(rows)

# Global reference line (full world population)
df_all = pd.read_csv(paths.der("population_by_macroregion_traveltime.csv"))
df_global = df_all.groupby('threshold_min')['population'].sum().reset_index()
df_global['pct'] = 100 * df_global['population'] / 7830089931
df_global = df_global[df_global['threshold_min'].isin(THRESHOLDS)].sort_values('threshold_min')

print("Coverage by income group (million people):")
print(df.groupby('income_group')['population'].sum().div(1e6).round(1))

In [ ]:
x = list(range(len(THRESHOLDS)))

fig, ax = plt.subplots(figsize=(8, 5))

for grp in INCOME_ORDER:
    sub = df_pct[df_pct['income_group'] == grp].sort_values('threshold_min')
    ax.plot(x, sub['pct'].values,
            color=INCOME_COLORS[grp], lw=2.5,
            label=INCOME_LABELS[grp], marker='o', ms=5, zorder=3)

ax.plot(x, df_global['pct'].values,
        color='black', lw=2, ls='--', label='Global', marker='s', ms=5, zorder=2)

# Dotted reference lines at 50% and 10-min threshold (index 2)
ax.axhline(50, color='gray', ls=':', lw=1, alpha=0.6)
ax.axvline(2, color='gray', ls=':', lw=1, alpha=0.6)

# Gap annotation at 10-min threshold
pct_high = df_pct[(df_pct['income_group']=='High') & (df_pct['threshold_min']==10)]['pct'].values[0]
pct_low  = df_pct[(df_pct['income_group']=='Low')  & (df_pct['threshold_min']==10)]['pct'].values[0]
ax.annotate('', xy=(2.08, pct_low + 0.5), xytext=(2.08, pct_high - 0.5),
            arrowprops=dict(arrowstyle='<->', color='#555555', lw=1.5))
ax.text(2.2, (pct_high + pct_low) / 2,
        f'Δ{pct_high - pct_low:.1f} pp', fontsize=10, color='#555555', va='center')

ax.set_xticks(x)
ax.set_xticklabels([str(t) for t in THRESHOLDS])
ax.set_xlabel('Travel time to nearest forest [min]', fontsize=13)
ax.set_ylabel('Population with access [%]', fontsize=13)
ax.set_ylim([0, 105])
ax.legend(fontsize=11, loc='lower right', frameon=True)
ax.text(0.01, 0.975, 'A', transform=ax.transAxes, fontsize=20, fontweight='bold', va='top')

fig.tight_layout()
fig.savefig(paths.fig("Figure4.pdf"), dpi=300, bbox_inches='tight')